# forkscope — 哪一步决定了 agent 的结果

输入一条 agent 轨迹 → 输出"**第 d 步把结局锁定为 X**"的归因报告,附反事实证据和成本核算。

**headline**(Qwen3-8B + SQL/calculator/search 工具环境,SGLang 原生):
- text-to-SQL 任务 t4:第一条 SQL 的写法把结局从"22% 正确"锁到 **0% 或 100%**——写法本身由 schema 措辞诱导(`Album.ArtistId` → 整数除法陷阱,46% 的 episode 掉进去,自愈率 2/126≈2%)
- 单位换算任务 t7:分叉**不在第一步**(98% 的 episode 首步 SQL 完全相同),决策步重放把它定位在第二步的"是否调 calculator 换算"上
- 这些都是**重采样直接测出的因果量**,不是 LLM 读日志的猜测(对照:Who&When,ICML'25——LLM judge 定位失败步准确率仅 14.2%)

## 0. 环境:真实工具,埋好陷阱,可执行的 gold

- **DB**:Chinook 音乐商店 SQLite(Artist 275 / Album 347 / Track 3503,11 表),查询真执行、只读
- **工具×3**(OpenAI function-calling,SGLang chat completions 下发):`sql_query`(描述里附 schema 文本——**陷阱来源**:列出 `Album.ArtistId` 会 prime `COUNT(DISTINCT ArtistId)` 写法)、`calculator`(正则白名单+沙箱)、`web_search`(**确定性 mock**,固定语料,里面故意放了诱饵数字)
- **10 个任务**全部带可执行 gold(`tasks.py::gold()` 对库直接算,无人工标注);陷阱谱系:schema-priming(t4)、陷阱表 `temp_country_codes`(t2)、off-by-one(t8)、诱饵数字(t10)
- **episode**:system(分析师人设+答案格式)→ 任务 → ≤8 轮 tool loop,temperature 1.0,per-episode seed,`--enable-deterministic-inference` 下逐 token 可复现
- **规模**:每任务 200 条独立 episode;决策步重放 K=50

一条真实的失败轨迹(seed 10000,`data/reports/transcripts.json` 原文)——全程两轮,**没有任何一处"看起来在犯错"**:

```
[assistant] tool_call: sql_query(
              "SELECT COUNT(*) / COUNT(DISTINCT ArtistId)
               AS AverageAlbumsPerArtist FROM Album;")    ← 分叉点
[tool]      {"rows": [{"AverageAlbumsPerArtist": 1}]}     ← 整数除法,合法返回,无错误
[assistant] "The average number of albums per artist ... is 1.000 ...
             The answer is 1.000."                        ← 自信收尾
```

这就是为什么 trace/log 工具看不见它:每一行单独看都是"正常"的。

## 1. t4 · 第一条 SQL 定生死

任务:"平均每个 artist 有几张 album?"(gold 1.701)。200 条独立 episode(temperature 1.0),按第一条 SQL 的写法分组:

| 第一条 SQL 写法 | 占比 | 最终正确率 |
|---|---|---|
| `AVG(...) GROUP BY`(对) | 18% | **100%** |
| `COUNT(*)/COUNT(DISTINCT ArtistId)`(整数除法,返回 1) | **46%** | **0%** |
| 错表 `FROM Artist`(275 vs 204) | 4% | 0% |
| 其它 | 33% | 39% |

MI(首步写法; 正确) = **0.568 bits**。这还只是相关性——下图才是因果:取一条失败轨迹和一条成功轨迹,在每个决策步边界重放 50 条完整续写(工具真执行),看结局分布 o_d 怎么变。

![t4 mirror](../data/reports/fig_t4_mirror.png)

同一个起点分布(o_0 = 22% 正确),intdiv 分支一步坍缩到 0%(persistence 1.00,50/50 续写全部沿着 1.000 走到黑),AVG 分支一步锁定 100%。**正确写法本身是个 16% 概率的少数派选择**(persistence 0.16)——这解释了为什么"多采几次"救不了它:掉进陷阱后模型没有任何错误信号(intdiv 执行成功、返回合法的 1)。

In [ ]:
import json

v = json.load(open("../data/reports/validation_t4_artist_album_ratio.json"))
print(f"episodes={v['n']}  gold={v['gold']}  overall_correct={v['overall_correct']:.1%}  "
      f"MI={v['mi_bits']:.3f} bits  decision_locked={v['decision_locked']}\n")
for c in v["clusters"]:
    print(f"  {c['cluster']:18s} n={c['n']:3d} ({c['share']:4.0%})  correct={c['correct_rate']:4.0%}  {c['outcomes']}")

## 2. t7 · 分叉不一定在第一步——重放能逐步定位它

任务:"平均 track 长度是几分钟?"(gold 6.56)。这里首步几乎没有分歧:**98% 的 episode 写同一条 `SELECT AVG(Milliseconds)`**(MI ≈ 0.02 bits),但 75% 的 episode 最后把秒当分钟答(393.6 vs 6.56)。失败发生在哪?对每个决策步边界重放:

## 3. 失败不是一种东西——分类学 + 三个干预点

| 类型 | 案例 | o_t 结构 | 干预点 |
|---|---|---|---|
| **知识层** | MMLU econometrics[6] | 全程自信地错,0 分叉 | 模型/prompt 层,轨迹内无救 |
| **漂移层** | MMLU virology[5] | B/C 高频拉锯,无单一落点 | 采样/聚合层(self-consistency) |
| **决策层·首步** | t4(schema 诱导) | d=0 锁定 | schema/工具描述措辞 |
| **决策层·中段** | t7(单位换算) | d=1 锁定 | 该步前的上下文措辞 |
| **诱饵锁定** | t10 | **187/200 抄了语料里的 67.3**(全流媒体)而非算 19.3/28.6=67.5 | 检索结果去混淆 |

t10 失败模式单一到 94%,与 t4 同族:**上下文里紧邻的错误候选会锁定结局**(t4 是 schema 列名,t10 是诱饵数字)。现有 observability(trace/log/LangSmith)全都看不出来;LLM judge 读日志(Who&When 方法族)step 级准确率 14-21%。分布级测量是唯一能把这些类型**自动分开**并给出干预点的路径。

漂移型的价值在"正确地拒绝":virology[5] 的 o_t 全程拉锯,平滑模型没有硬找出假分叉——

In [ ]:
import json

for tag, path in [
    ("t4 fail (intdiv)", "../data/reports/replay_t4_artist_album_ratio_sql_intdiv_10000.json"),
    ("t4 ok  (avg)    ", "../data/reports/replay_t4_artist_album_ratio_sql_avg_groupby_10015.json"),
    ("t7 fail         ", "../data/reports/replay_t7_avg_track_len_min_sql_avg_ms_raw_10000.json"),
    ("t7 ok           ", "../data/reports/replay_t7_avg_track_len_min_sql_avg_ms_raw_10003.json"),
]:
    r = json.load(open(path))
    curve = " -> ".join(f"{s['p_correct']:.2f}" for s in r["steps"])
    print(f"{tag}  o_d[correct]: {curve}")
    for s in r["steps"]:
        print(f"    d={s['d']}: recorded={s['recorded_next']:24s} persistence={s['persistence']:.2f}  o_d={s['o_d']}")

## 4. 闭环:agent 读测量报告,修好了自己——以及"只给定位"够不够

把 t4 的分叉归因报告(重采样测出的统计事实,**不是它自己的日志**)交给同一个模型改写工具描述,200 vs 200 重测;再做**去泄露对照**(agent 只写 ≤60 词警告便签,代码 append,结构上无法夹带正确写法):

| arm | t4 correct | t4 首步错表 | t7 correct |
|---|---|---|---|
| control | 27.0% | 9.5% | 17.0% |
| 报告+改法(含正确模式) | **100%** | 0% | — |
| 只警告(去泄露便签) | **15.5% ↓** | **32.5% ↑** | **37.5%(2.2×)↑** |

三层结论:**t7 纯警告有效**(修复="检查单位",模型本来就会做——定位本身值钱);**t4 纯警告有害**(intdiv 被抑制但错误位移到错表,打地鼠);**完整因果报告才到 100%**。测量的价值分层:定位是下限,因果机制授权的建设性修复是上限。对照 Who&When:LLM 读日志连定位都只有 14.2%。

过程教训:第一版约束"只能警告"→ 模型删光 schema → 81-97% episode 瘫痪。**RSI 的 patch 应用必须结构受限(append-only)**——对 DGM 类系统的普适教训。(数据:`rsi_loop.json` / `rsi_noleak2.json`;各臂 rsi_loop 口径,臂间对比有效。)

## 方法一页纸:采样、检验、优化

1. **估计量** o_d = P(结局 | 前缀到 d),S 条 iid 续写的经验分布;相邻 o_d 的 TVD 跳变 = 分叉。重放在完整 chat 分布(template + tool schema)下、工具真执行——on-policy。
2. **噪声模型** Multinomial(S, o_d),replicate-TVD ∝ 1/√S(t4 实测斜率 −0.493);"是不是真分叉"对 exact iid multinomial null 检验,T=1 用模拟 null 的 p 值;seed 段重批排除坏批。
3. **方差压缩**(token 级)PELT 变点 + 段内核加权 Dirichlet pooling,等效样本 3.8×;决策步粒度不需要。
4. **系统优化** RadixAttention 前缀共享(实测 prefill ↓14.9×);每分支独立请求 + seed=f(任务,步,序号);deterministic inference → 全 pipeline 逐 token 可复现。
5. **persistence**(= Thought Branches 的 resilience 之决策步版):区分"分叉"(低 persistence、结局改变)与"必经之路"(persistence 1.0)。

## 3. 失败不是一种东西——三种类型,三个干预点

| 类型 | 案例 | o_t 结构 | 干预点 |
|---|---|---|---|
| **知识层** | MMLU econometrics[6] | 全程自信地错,0 分叉 | 模型/prompt 层,轨迹内无救 |
| **漂移层** | MMLU virology[5] | B/C 高频拉锯,无单一落点 | 采样/聚合层(self-consistency) |
| **决策层** | **agent t4 / t7** | **某一步把 o_t 锁死** | **该步的写法/工具选择——FPA 可精确定位** |
| └ 首步锁定 | t4(schema 诱导) | d=0 分叉 | schema/工具描述措辞 |
| └ 中段锁定 | t7(单位换算) | d=1 分叉 | 该步前的上下文措辞 |

现有 observability(trace/log/LangSmith)三种都看不出来;LLM judge 读日志(Who&When 方法族)step 级准确率 14-21%。分布级测量是唯一能把三类**自动分开**并给出干预点的路径。

漂移型的价值在"正确地拒绝":virology[5] 的 o_t 全程拉锯,平滑模型没有硬找出假分叉——

![virology triptych](../data/reports/virology_5_triptych.png)

## 4. 统计有效性(平滑模型的前提假设)

- **V1 多项式噪声**:agent 结局的 replicate-TVD 对 exact iid multinomial null 做检验——t4 p=0.28、t7 p=0.09、t10 p=0.22,**无法拒绝多项式假设**(单决策点 S=200;同前缀重批 3×50 的 seed 段检查无异常段)。
- **V2 1/√S 律**:t4 log-log 斜率 **−0.493**(理论 −0.5,论文实测 −0.49),verdict supported。
- **V3 平滑恢复**:合成数据上 22/22 测试通过,TVD 降 36%,等效样本量 **3.8×**;真实 MCQ 数据上的表现是"正确地拒绝"(virology 无假分叉)。
- watch-item:t7 同前缀两批之间有 ~1.9σ 的轻度过散,持续监控中(重放多位置数据攒够后用大 T 重验)。

In [ ]:
import json

s = json.load(open("../data/reports/agent_v1v2.json"))
for tid, r in s.items():
    if r.get("degenerate"):
        print(f"{tid}: degenerate (single outcome)")
        continue
    v1, v2 = r["v1"], r["v2"]
    print(f"{tid}: V1 measured={v1['measured']:.3f} null={v1['null']:.3f}  "
          f"V2 slope={v2.get('slope', float('nan')):.3f} verdict={v2['verdict']}")

## 5. 成本 · 为什么要在 serving 引擎里做

FPA 的成本结构 = 大量共享前缀的重采样。RadixAttention 让嵌套前缀的 prefill 几乎免费:

- 全 pipeline cache 命中率 **98.5%**(gsm8k smoke,112K prefill 中 110K cached)
- 同一 t4 重放负载的 radix on/off **实测 A/B**(相同 seeds,decode 逐 token 一致):**prefill 计算量 ↓14.9×**,见下

对比:VinePPO(ICML'25)每步 fork K=9 条 MC 续写做 credit assignment,用 vLLM 无前缀共享,单步慢 2-5×;所有 PRM 标注管线(Math-Shepherd/OmegaPRM)的 MC 续写全部离线、无 serving 层优化。**o_t[correct] 在数学上就是 VinePPO 的 V(s_t)——我们把同一个量变成了 serving 引擎原生的 observability 输出。**

In [ ]:
# radix on/off 实测 A/B — 同一 t4 决策步重放负载 (2 boundaries x K=50, 相同 seeds)
# givemeanode h100-1, SGLang 0.5.18, /metrics 前后快照差值; decode 两侧逐 token 一致 (deterministic seeds)
ab = {
    "workload": "t4 replay: 2 boundaries x 50 continuations, identical seeds",
    "prompt tokens submitted (both)": "106.7K",
    "prefill computed, radix ON": "7.1K  (99.5K cached = 93.3% hit)",
    "prefill computed, radix OFF": "106.7K",
    "prefill compute reduction": "14.9x (measured, not estimated)",
    "decode tokens (both)": "4.7K",
    "wall-clock": "6s vs 8s (toy-scale prefixes; the gap grows with trajectory length)",
}
for k, v in ab.items():
    print(f"{k}: {v}")

## 6. 定位与已知边界

**相对已有工作,forkscope 占的空位:**
- Who&When / AgenTracer 家族:LLM-judge 或单次重放读**多 agent 日志**(step 级 14-21%);我们对**单 agent 轨迹**输出重采样分布 o_t + 失败三分类,有 ground truth
- Forking Paths / Thought Anchors / Thought Branches:token/句子级、MCQ/CoT、HF/API 采样;我们是 agent 工具轨迹 + serving 原生,且首次记录 **chat 分布 vs 裸续写的分布错配**(同一模型裸续写 100% 写对 SQL——schema 本身就是陷阱来源,裸续写测的是另一个模型)
- VinePPO / PRM:同一个 MC 量用于训练;我们用于诊断,不训 reward model,天然免疫 overcredit
- RSI(Darwin Gödel Machine):其自改进信号是"LLM 读 eval log"——正是 Who&When 证明只有 14% 准确率的环节;分叉点 + 反事实证据是更强且**不可被 agent 博弈**的信号源(重采样度量在 agent 外部)

**已知边界(诚实声明):**
- 规模:1 个 8B 模型、10 任务工具环境、决策锁定案例 2 个(t4 首步、t7 中段);"决策型分叉普遍存在"还需更多任务/模型
- t7 同前缀批间 ~1.9σ 轻度过散,监控中
- 平滑模型在 agent 决策步粒度还没跑(决策步太少,平滑是 token 级需求);token 级平滑只在合成 + MCQ 数据验证
- 决策型分叉在 8B MCQ 上 ~1% 稀有(201 题探针)——FPA 的自然栖息地是 agent 任务,MCQ 结论不外推